In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.table import Table
from gliders.utils import sigma_clipping, grow_mask, get_profile_from_border, normalize_data
from gliders.artifacts import detect_artifacts, get_artifact_flag, clean_artifacts, detect_and_clean_artifacts
from gliders.background_suppressor import suppress_background, show_all_methods
from scipy.stats import norm
from scipy.signal import convolve2d
from astropy.convolution import Gaussian2DKernel
import warnings
from astropy.utils.exceptions import AstropyDeprecationWarning
from photutils.utils.exceptions import NoDetectionsWarning
warnings.simplefilter('ignore', category=AstropyDeprecationWarning)
warnings.simplefilter('ignore', category=NoDetectionsWarning)

def extract_band_data(hdu_list, verbose=True, bands_to_analyze = ['r']):
    all_bands = {}
    all_img_diffs = {}
    all_img_areas = {}
    all_weights = {}
    all_wgh_diffs = {}
    all_wgh_areas = {}
    ref_hdu = None
    for hdu in hdu_list:
        try:
            header = hdu._header
            if verbose:
                print(header)
            band = header.get("BAND")
            if band not in bands_to_analyze: continue
            image_type = header.get("IMAGETYP")
            imagew = header.get("IMAGEW")
            imageh = header.get("IMAGEH")

            if band is None or image_type is None or imagew is None or imageh is None:
                continue  # Skip if essential header info is missing

            img_dim_diff = abs(imagew - imageh)
            img_area = imagew * imageh

            if image_type == "image":
                if band not in all_bands or img_area > all_img_areas[band] or (img_area == all_img_areas[band] and img_dim_diff < all_img_diffs[band]):
                    all_bands[band] = hdu.copy().data
                    all_img_areas[band] = img_area
                    all_img_diffs[band] = img_dim_diff
                    ref_hdu = hdu.header

            elif image_type == "invvar":
                w_map = hdu.copy().data
                w_map[w_map == 0] += 1  # Avoid division by zero
                weight = 1 / np.sqrt(w_map)
                if band not in all_weights or img_area > all_wgh_areas[band] or (img_area == all_wgh_areas[band] and img_dim_diff < all_wgh_diffs[band]):
                    all_weights[band] = weight
                    all_wgh_areas[band] = img_area
                    all_wgh_diffs[band] = img_dim_diff

        except Exception as e:
            print(f"Skipped HDU due to error: {e}")

    return all_bands, all_weights, ref_hdu

file_dir = "C:/Astrostuff/PhD/Data/DESI FITS/"
artif_file = "C:/Astrostuff/PhD/Data/My_data/DESI_New_mergers.csv"
artif_catalog = Table.read(artif_file, format = "csv", converters = {"object_id":str})
print(len(artif_catalog))
print(artif_catalog)

3692
    object_id         ra      dec       z    ...    I    merger_predicted  ES
----------------- --------- -------- ------- ... ------- ---------------- ---
39627902957912363 211.09828  4.71795 0.02977 ... 0.41312                1   1
39632991730798910 253.11914 35.41754 0.05766 ...  0.2024                1   1
39627829528234714 150.37899  1.68886 0.09869 ... 0.01352                1   1
39627830664894213  218.2308  1.86938 0.02998 ... 0.58443                1   1
39633427988745611 271.28566 63.42304  0.0915 ...  0.2459                1   1
39627758090851335  212.4838 -1.23884  0.0248 ... 0.47045                1   1
39633104083616306 152.75604 41.58966 0.05861 ... 0.60419                1   1
39627877821449269 150.00236  3.67734 0.07391 ... 0.03693                1   1
39627773882401920   73.5449 -0.39993 0.05467 ... 0.15411                1   1
39633321931571417 185.66454 55.08292 0.03664 ... 0.28208                1   1
              ...       ...      ...     ... ...     ...   

In [5]:
from photutils.profiles import RadialProfile
from scipy.stats import pearsonr
from astropy.modeling import models, fitting
from gliders.utils import get_image_centroid, sigma_clipping, grow_pixel_circular, get_profile_gradient
from gliders.star_filter import _get_distance_to_closest_edge, _compare_profiles, _extract_all_profiles, _patch_profile, _get_mean_profile_symmetry_ratio_sum
def process_one_star(this_star, image, params):
    """
    Process a single star to determine its radius and profile standard deviation.

    Parameters
    ----------
    this_star : dict
        A dictionary containing information about the star, including its centroid coordinates.
    image : 2D array
        The image in which the star is located.
    params : tuple
        A tuple containing various parameters for processing, including:
        - min_radius : int
        - max_radius : int
        - rp_psf : RadialProfile or None
        - rp_error : 2D array
        - image_background : float
        - gaussian_fitter : astropy.modeling.fitting.Fitter
        - std_convergence_threshold : float
            The relative difference threshold for the convergence of the fitted gaussian profile std. Determines if the
            star radius is compared to the fitted gaussian std (see gaussian_fit_max_sigma).
        - gaussian_fit_max_sigma : float
            The star radius cannot be larger than the fitted gaussian profile std * gaussian_fit_max_sigma. This is
            only checked if the relative difference between the profiles of two consecutive radii is lower than
            std_convergence_threshold
        - slope_threshold : float
        - delta_slope_threshold : float
        - profile_cos_distance_threshold : float
        - mean_symmetry_ratio_threshold : float
        - gaussian_ratio_threshold : float
        - fill_value : float

    Returns
    -------
    final_radius : int
        The determined radius of the star. Returns 0 if the star does not pass the filtering criteria.
    final_std : float
        The standard deviation of the star's profile. Returns 0 if the star does not pass the filtering criteria.
    """

    (
        min_radius, max_radius, rp_psf, rp_error,
        image_background, gaussian_fitter, std_convergence_threshold, gaussian_fit_max_sigma,
        slope_threshold, delta_slope_threshold,
        profile_cos_distance_threshold, mean_symmetry_ratio_threshold,
        gaussian_ratio_threshold, fill_value
    ) = params

    y_cen, x_cen = this_star['y_centroid'], this_star['x_centroid']
    candidate_pos = (int(round(y_cen)), int(round(x_cen)))
    candidate_eff_pos = candidate_pos
    # Clip centroid so it is always within the image.
    candidate_eff_pos = (np.clip(candidate_eff_pos[0], 0, image.shape[0] - 1),
                          np.clip(candidate_eff_pos[1], 0, image.shape[1] - 1))
    distance_to_closest_edge = _get_distance_to_closest_edge(image.shape, candidate_eff_pos)
    relocated = False
    final_radius = 0
    final_profile = None
    final_std = 0
    profile_std = None
    half_image_min = np.min(image.shape) / 2

    # ----------------------------------------------------------
    # MAIN RADIUS LOOP
    # ----------------------------------------------------------
    for radius in range(min_radius, max_radius + 1):
        if radius > half_image_min:
            break
        if distance_to_closest_edge < radius / 2 and not relocated:
            mask = grow_pixel_circular(image.shape, candidate_pos, radius)
            # new_cc = get_image_centroid(image * mask) # TODO: Check and test if this is better than the below method
            new_cc = get_image_centroid(image * image * mask)
            if new_cc is not None:
                new_c_x, new_c_y = new_cc
                candidate_eff_pos = (new_c_y, new_c_x)
                relocated = True

        edge_radii = np.arange(radius + 1)
        rp = RadialProfile(image, (candidate_eff_pos[1], candidate_eff_pos[0]),
                           edge_radii, error=rp_error, mask=None, method="center")

        if np.sum(np.abs(rp.profile)) == 0:
            break
        if np.any(rp.data_profile <= image_background):
            break

        new_profile_std = rp.gaussian_fit.stddev.value
        if profile_std is not None:
            if profile_std == 0:
                diff = std_convergence_threshold + 1
            else:
                diff = abs(profile_std - new_profile_std) / profile_std
            if diff < std_convergence_threshold:
                if radius > profile_std * gaussian_fit_max_sigma:
                    break
        profile_std = new_profile_std

        print("rp.data_profile")
        print(rp.data_profile)
        if len(rp.data_profile) < 2:
            continue

        profile_slopes, profile_delta_slopes = get_profile_gradient(rp.profile)
        max_delta_slope = np.max(profile_delta_slopes)

        if len(profile_delta_slopes) > 1:
            slope_last = profile_slopes[-1]
            slope_prev = profile_slopes[-2]
            delta_last = profile_delta_slopes[-1]
            delta_prev = profile_delta_slopes[-2]

            if slope_last > slope_prev and np.max(profile_slopes) > slope_threshold: # TODO: Check why this.
                if delta_last < delta_prev and delta_last < delta_slope_threshold:
                    break

        if max_delta_slope > 0: # TODO: Check why this.
            final_radius = radius
            final_profile = rp.profile
            final_std = profile_std


    # ----------------------------------------------------------
    # POST-PROCESSING IF STAR PASSED RADIUS LOOP
    # ----------------------------------------------------------
    if final_radius <= 0:
        return final_radius, final_std
    else:
        if this_star['type'] != "PSF" and rp_psf is not None:
            psf_prof = rp_psf.profile[:len(final_profile)]
            if len(psf_prof) < len(final_profile):
                tail = np.full((len(final_profile) - len(psf_prof),), psf_prof[-1])
                psf_prof = np.append(psf_prof, tail)
            corr, pval, euc_dist, cos_dist = _compare_profiles(psf_prof, final_profile)
            if cos_dist > profile_cos_distance_threshold:
                return 0, 0

        # symmetry tests
        y0, x0 = candidate_eff_pos
        ny, nx = image.shape
        cutout = image[
            max(y0 - final_radius, 0):min(y0 + final_radius + 1, ny),
            max(x0 - final_radius, 0):min(x0 + final_radius + 1, nx)
        ]

        cutout_ny, cutout_nx  = cutout.shape
        cutout_y, cutout_x = np.indices(cutout.shape)
        init = models.Gaussian2D(amplitude=cutout.max(),
                                 x_mean=cutout_nx / 2, y_mean=cutout_ny / 2,
                                 x_stddev=final_radius / 2, y_stddev=final_radius / 2)
        init.x_mean.bounds = (0, cutout_nx)
        init.y_mean.bounds = (0, cutout_ny)

        fit = gaussian_fitter(init, cutout_x, cutout_y, cutout)
        x_std = fit.x_stddev.value
        y_std = fit.y_stddev.value

        if (not np.isfinite(x_std) or not np.isfinite(y_std) or x_std <= 0 or y_std <= 0):
            return 0, 0
        gaussian_ratio = max(x_std / y_std, y_std / x_std)

        mask = grow_pixel_circular(image.shape, candidate_eff_pos, final_radius)
        profiles = _extract_all_profiles(image, mask, candidate_eff_pos, final_radius, fill_value)
        for key in profiles:
            profiles[key] = _patch_profile(profiles[key], fill_value)

        mean_symmetry_ratio = _get_mean_profile_symmetry_ratio_sum(profiles, fill_value)
        if (mean_symmetry_ratio > mean_symmetry_ratio_threshold or
                gaussian_ratio > gaussian_ratio_threshold):
            return 0, 0

        # Success — update and return the result
        return final_radius, final_std

In [7]:
bleed_radii = []
areas = []
s_detections = {'fwhm':[], 'kernel_std':[], 'detections':[], 'source':[]}
from gliders.cleaning_configs import configs
from astropy.modeling import models, fitting
import pandas as pd
config = configs["default"]
for ii, object_info in enumerate(artif_catalog[140:]):
    if ii <= 0:
        continue
    object_id = object_info[0]
    # print(ii, object_id)
    obj_coords = (object_info[1],object_info[2])
    redshift = object_info[3]
    bands = ['r'] #,'g']
    file_name = f"{object_id}_{"".join(bands)}.fits"
    sources_name = f"{object_id}_sources.fits"

    file_path = os.path.join(file_dir, file_name)
    sources_path = os.path.join(file_dir, sources_name)
    try:
        with fits.open(file_path) as hdu_list:
            # hdu_list.info()
            all_bands, all_weights, ref_header = extract_band_data(hdu_list, verbose=False, bands_to_analyze=bands)
    except:
        continue

    with fits.open(sources_path) as hdul:
        data = hdul[1].data
        as_table = Table(data)
    wcs = WCS(ref_header)
    tractor_x_pix, tractor_y_pix = wcs.all_world2pix(as_table['ra'], as_table['dec'], 0)  # origin=0 for Python indexing
    source_x_pix, source_y_pix = wcs.all_world2pix(obj_coords[0], obj_coords[1], 0)  # origin=0 for Python indexing
    source_pos = (np.round(source_x_pix), np.round(source_y_pix))
    as_table['x_centroid'] = tractor_x_pix
    as_table['y_centroid'] = tractor_y_pix
    phot_sources = as_table['type','x_centroid','y_centroid']
    image = all_bands['r']


    if obj_coords[1] > 32.375:
        pix_scale = 1.72
    else:
        pix_scale = 1

    from photutils.segmentation import SourceCatalog, detect_threshold, detect_sources
    # from gliders.background_suppressor import suppress_background
    from gliders.stars import detect_and_mask_stars, find_all_star_apertures, plot_apertures, classify_apertures, detect_stars_in_image, get_star_apertures
    from astropy.table import vstack, hstack
    source_min_pix = 10
    # image = suppress_background(image, method = "layers")
    from gliders.pipeline import update_detections
    from gliders.utils import convolve_with_gaussian_kernel, smooth_blend
    from photutils.segmentation import SegmentationImage
    from astropy.stats import SigmaClip

    # image = suppress_background(image, method = "layers")
    image_u, seg_map, main_source_segmap = update_detections(image,source_min_pix=20, source_pos=source_pos, use_filter=True)

    main_source_mask = main_source_segmap.make_source_mask()
    main_source_npixels = np.sum(main_source_mask)

    # safe_star_masking_radius = round(np.sqrt(main_source_npixels * config.safe_star_masking_prop_radius / np.pi))
    # print(safe_star_masking_radius)



    # safe_star_masking_radius = 0
    # all_star_apertures = find_all_star_apertures(image, source_pos, phot_sources, safe_star_masking_radius, config, use_phot_sources = True, use_finders = False, clear_nucleus = False, image_psf = None)

    all_stars_detected_list = []

    if phot_sources is not None:
        # We use the detections from the input use_phot_sources table
        stars_from_phot_sources = phot_sources[phot_sources['type']=='PSF']
        all_stars_detected_list.append(stars_from_phot_sources)
    print(all_stars_detected_list)

    if len(all_stars_detected_list) > 0:
        all_stars_detected = vstack(all_stars_detected_list, metadata_conflicts = 'silent')
        all_stars_detected['id'] = np.arange(1, len(all_stars_detected) + 1)
        all_star_apertures = get_star_apertures(all_stars_detected)
        all_star_apertures_split = classify_apertures(all_star_apertures, main_source_mask)
        # plot_apertures(image, all_star_apertures_split)
    print(all_star_apertures)
    stars_detected = all_star_apertures

    image_background, image_background_std = sigma_clipping(image)
    rp_error = np.full_like(image, image_background_std)

    min_radius = 2
    max_radius = 10
    rp_psf = None
    gaussian_fitter = fitting.TRFLSQFitter()
    slope_threshold = -0.008
    delta_slope_threshold = 0.008
    profile_cos_distance_threshold = 0.045
    mean_symmetry_ratio_threshold = 0.07
    gaussian_ratio_threshold = 1.2
    std_convergence_threshold = 0.01
    gaussian_fit_max_sigma = 3
    fill_value = -999

    params = (
        min_radius, max_radius, rp_psf, rp_error,
        image_background, gaussian_fitter,
        std_convergence_threshold, gaussian_fit_max_sigma,
        slope_threshold, delta_slope_threshold,
        profile_cos_distance_threshold, mean_symmetry_ratio_threshold,
        gaussian_ratio_threshold, fill_value
    )


    for this_star in stars_detected:
        print(this_star)
        final_radius, final_std = process_one_star(this_star, image, params)
        break
    print(final_radius, final_std)
    break


    # print(df)
    # break
    # print(all_stars_detected_list)

    # if len(all_stars_detected_list) > 0:
    #     all_stars_detected = vstack(all_stars_detected_list, metadata_conflicts = 'silent')
    #     all_stars_detected['id'] = np.arange(1, len(all_stars_detected) + 1)
    #     all_star_apertures = get_star_apertures(all_stars_detected)
    #
    #
    #     all_star_apertures_split = classify_apertures(all_star_apertures, main_source_mask)
    #     plot_apertures(image, all_star_apertures_split)
    # print(all_star_apertures)

    # plt.imshow(main_source_mask)
    # plt.show()

    # if config.auto_safe_masking_region:
    #     safe_star_masking_radius = round(np.sqrt(main_source_npixels * config.safe_star_masking_prop_radius / np.pi))
    #     if safe_star_masking_radius < 1: safe_star_masking_radius = 1
    # else:
    #     safe_star_masking_radius = config.safe_star_masking_radius
    #
    # all_star_apertures = find_all_star_apertures(image, main_source_pos, phot_sources, safe_star_masking_radius, config, use_phot_sources = True, use_finders = False, clear_nucleus = True, image_psf = image_psf)
    print(ii)
    if ii >= 30:
        break
# df = pd.DataFrame(s_detections)
# bleed_radii = np.array(bleed_radii)
# areas = np.array(areas)
# df.to_csv("C:\\Astrostuff\\PhD\\Data\\star_detections.csv", index = False)
print("DONE!")

[<Table length=31>
type     x_centroid         y_centroid    
str3      float64            float64      
---- ------------------ ------------------
 PSF   395.261105865827 30.686934791177123
 PSF  392.0664632960463 22.246528407821984
 PSF 379.62518019999266 344.16124018031974
 PSF  375.8404661352829  147.8750390858604
 PSF  374.2629219606852  170.3128377795383
 PSF 363.71230672968096  364.4404063226966
 PSF  351.4598827007936 136.47491351526605
 PSF 350.45015939058413  66.76405121889661
 PSF  324.5931558408864 134.35030423339708
 ...                ...                ...
 PSF 159.04074238339692  61.14842429016994
 PSF  132.9843295542944 250.39589201448575
 PSF 116.36177223438705  235.7003637406392
 PSF  85.47099129254107  7.135094863466293
 PSF  79.82344941701501 108.70324839911927
 PSF  74.59822657066249  92.13116424428358
 PSF 48.315585553817755 267.98461871868494
 PSF  35.77801734393998  18.80223187205786
 PSF 15.385341749297424  153.0139982717019
 PSF  5.333150032968476  89.0002436

TypeError: string indices must be integers, not 'str'